# 使用 Bedrock AgentCore Identity 安全管理凭证

本动手实验演示如何将 Strands Agents 与 Amazon Bedrock AgentCore Identity 集成，以便在构建 AI 代理时安全管理外部服务的 API 密钥和凭证。

## 概述

在本实验中，您将：
- 了解安全凭证管理的挑战
- 理解 Bedrock AgentCore Identity 的功能
- 为外部服务创建 API Key Credential Provider
- 测试 AI 代理中的安全凭证检索
- 探索凭证管理的最佳实践

## 前提条件

在开始本实验之前，请确保您已具备以下条件：
- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 包
- 基于 AWS 区域的 Nova Pro 模型 ID
- 用于测试的外部 API 密钥（例如 Exa API 密钥）

如果您未在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

安装 Strands Agents 和 Bedrock AgentCore Python SDK 所需的包：

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## 获取 Exa API 密钥以连接远程 Exa MCP

在本实验中，我们将与 [Remote Exa MCP](https://docs.exa.ai/reference/exa-mcp) 交互，通过 Exa Search API 执行实时网络搜索，这需要 Exa API 密钥才能连接。

Exa MCP 服务器 URL：```https://mcp.exa.ai/mcp?exaApiKey=your-exa-api-key```

要获取 Exa API 密钥，请前往 [Exa 登录页面](https://dashboard.exa.ai/login) 使用您的电子邮件注册。

然后前往 Exa 控制面板中的 [API 密钥部分](https://dashboard.exa.ai/api-keys) 创建 API 密钥。将 API 密钥复制到下方代码中的 `EXA_API_KEY`...

## 理解凭证管理的挑战

### 演示不安全的 API 密钥使用方式

让我们首先演示在没有适当凭证管理的情况下尝试使用外部服务（Exa 搜索）时会发生什么。这将展示硬编码或无效 API 密钥带来的安全风险和身份验证失败。

⚠️ **重要提示**：请将占位符 API 密钥替换为您的实际 Exa API 密钥。

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

# !-------- UPDATE THE EXA API KEY HERE  --------!
EXA_API_KEY = <YOUR EXA API KEY> 

# Connect to the weather MCP server
print("\nConnecting to MCP Server...")
exa_server = MCPClient(lambda: streamablehttp_client(f"https://mcp.exa.ai/mcp?exaApiKey={EXA_API_KEY}"))

with exa_server:
    # Combine all tools - they all work the same way!
    mcp_tools = (
        exa_server.list_tools_sync()
    )

    print(f"Available tools: {[tool.tool_name for tool in mcp_tools]}")
    
    # Create agent with Exa MCP tools
    agent = Agent(model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
                  system_prompt="你是一个生活助手，运用网络搜索的知识回答各种问题。",
                  tools=mcp_tools)

    agent("什么是 Amazon Bedrock AgentCore？")

## 什么是 Bedrock AgentCore Identity？

Amazon Bedrock AgentCore Identity 为需要访问外部服务的 AI 代理提供安全的凭证管理。主要优势包括：

- **安全存储**：在 AWS Secrets Manager 中存储 API 密钥、令牌和凭证
- **运行时检索**：在运行时提供安全的凭证访问，无需硬编码
- **访问控制**：与 AWS IAM 集成，实现细粒度的访问权限管理
- **审计追踪**：维护凭证访问日志，用于安全监控
- **轮换支持**：支持自动凭证轮换和生命周期管理

该服务消除了在应用程序代码中硬编码敏感凭证的需要，降低了安全风险并提高了合规性。

## 创建安全凭证管理

### 步骤 1：创建 API Key Credential Provider

现在我们将使用 Bedrock AgentCore Identity 创建一个 API Key Credential Provider。这会将 Exa API 密钥安全地存储在 AWS Secrets Manager 中，并提供一种托管方式来访问它，而无需在代码中暴露凭证。

⚠️ **重要提示**：请将占位符 API 密钥替换为您的实际 Exa API 密钥。

In [ ]:
from bedrock_agentcore.services.identity import IdentityClient
from botocore.exceptions import ClientError
import boto3

# !-------- UPDATE THE EXA API KEY HERE  --------!
EXA_API_KEY = <YOUR EXA API KEY> 

region = boto3.session.Session().region_name

#Configure API Key Provider
identity_client = IdentityClient(region=region)

try:
    api_key_provider = identity_client.create_api_key_credential_provider({
        "name": "exa-apikey-provider",
        "apiKey": EXA_API_KEY # Replace it with the API key you obtain from the external application vendor, e.g., OpenAI
    })
    print("Created AgentCore Identity API Key Credential Provider.")
    print(api_key_provider)
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        print("AgentCore Identity API Key Credential Provider already exist.")
    else:
        print(f"ERROR: {e}")
except Exception as e:
    # Show any errors during api key provider creation
    print(f"ERROR: {e}")

### 步骤 2：测试安全凭证检索

现在让我们使用安全凭证检索来测试代理。`@requires_api_key` 装饰器会在运行时自动从凭证提供程序检索 API 密钥，确保代码中没有硬编码的密钥，同时保持安全最佳实践。

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.identity.auth import requires_api_key

@requires_api_key(provider_name="exa-apikey-provider")
def need_api_key(*, api_key: str):
    print(f'received api key for async func: {api_key}')
    return api_key

EXA_API_KEY = need_api_key()

# Connect to the EXA MCP server
print("\nConnecting to MCP Server...")
exa_server = MCPClient(lambda: streamablehttp_client(f"https://mcp.exa.ai/mcp?exaApiKey={EXA_API_KEY}"))

with exa_server:
    # Combine all tools - they all work the same way!
    mcp_tools = (
        exa_server.list_tools_sync()
    )

    print(f"Available tools: {[tool.tool_name for tool in mcp_tools]}")
    
    # Create agent with Exa MCP tools
    agent = Agent(model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
                  system_prompt="你是一个生活助手，运用网络搜索的知识回答各种问题。",
                  tools=mcp_tools)

    agent("什么是 Amazon Bedrock AgentCore？")

让我们查看代理循环的详细执行流程，以了解代理如何处理请求并生成响应：

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

## 资源清理（可选）

清理已部署的资源：

In [ ]:
import boto3

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)

try:
    print("Deleting AgentCore Identity...")
    agentcore_control_client.delete_api_key_credential_provider(name="exa-apikey-provider")
    print("✓ AgentCore Identity deletion initiated")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

## 总结

在本实验中，您成功完成了以下内容：

- ✅ 识别了代理应用程序中硬编码 API 密钥的安全风险
- ✅ 使用 Bedrock AgentCore Identity 创建了安全的 API Key Credential Provider
- ✅ 实现了用于外部服务集成的安全凭证检索
- ✅ 使用 Exa API 测试了安全凭证访问以实现网络搜索功能

## AgentCore Identity 的主要优势

- **安全凭证存储**：对 API 密钥和密钥进行加密存储
- **访问控制**：细粒度的凭证访问权限管理
- **审计追踪**：完整记录凭证使用和访问日志
- **集成就绪**：与 MCP 服务器和代理工作流无缝集成
- **最佳实践**：消除硬编码凭证和安全漏洞
